# Example notebook
1. `conda env create -f env.yml` will create `wilsonenv` environment
2. *pip install* `Wilson` and `CQCParse` packages: inside respective directories `pip install .` or `pip install -e .` (for future repo updates), probably in a new environment. 
3. Test installation: copy this notebook outside of `Wilson` directory and run in the `wilsonenv`.

In [ ]:
from wilson.spectrum.spectrum2D import Spectrum2D
import numpy as np
# np.set_printoptions(precision=4)

from CQCParse.parsing import GaussianDataParser, CFOURdataParser
from CQCParse.relay import DataVault
from wilson.utils import get_package_root

In [ ]:
wilson_root = get_package_root()
data_vault = DataVault(wilson_root+'/tests/test_database/mini_files_database.csv')
# data_vault = DataVault('/mnt/c/Users/vle014/OneDrive - UiT Office 365/Documents/files_fram/files_database.csv')

dataframe_gaussian = data_vault.getting_files_DB("gaussian")
method_basis = dataframe_gaussian[(dataframe_gaussian['code'] == 'ACAC') & (dataframe_gaussian['method'] != 'PBE0')][["code", "method", "basis_set"]]
tuples_method_basis = [(row['code'], row['method'], row['basis_set']) for index, row in method_basis.iterrows()]
dataframe_gaussian

In [ ]:
# omega1 = np.arange(1280., 3150., 235.41)
# omega2 = np.arange(1589., 6050., 465.41)
omega1 = np.arange(1280., 3150., 35.41)
omega2 = np.arange(1589., 6050., 65.41)

datadict = data_vault.make_DatainputDict('gaussian', ('FORM', 'B3LYP', 'cc_pVDZ'), wilson_root)
dictInputs = {'parserObject': GaussianDataParser(datadict), 'el_terms_select': [0,1], 'mech_terms_select': [0,]}

# ------- setting up a Spectrum2D object
spectrumObj = Spectrum2D(omega1, omega2)
spectrumObj.load_data(dictInputs['parserObject'])
spectrumObj.set_spectrum_settings(Gamma_rc=10., diag_margin_rc=10., vib_levels_harmonic=False)
spectrumObj.add_terms(dictInputs['el_terms_select'], dictInputs['mech_terms_select']) # currently requires diag_margin_rc attribute to be set
spectrumObj.precalculate4fullspectrum()

# print('spectrumObj.mech_avrg_tensors')
# for i in spectrumObj.mech_avrg_tensors:
#     print(repr(i), '\n')
    
print('before computedSpectrum.fundamentals')
print(spectrumObj.fundamentals)
print(sorted(list(spectrumObj.fundamentals.values())))
print('\ncomputedSpectrum.all_states\n', spectrumObj.all_states)

# ------- computing anharmonicities
sec_hypol_dataALL = 0

sec_hypol_data1 = 0
if dictInputs['mech_terms_select']:
    electrical, Qab_contrib_dict = spectrumObj.intensity_electrical()
    sec_hypol_data1 += electrical
    # print(f'\nElectrical {dictInputs["el_terms_select"]}')
    # print(repr(sec_hypol_data1))

sec_hypol_dataALL+=sec_hypol_data1

sec_hypol_data2 = 0
if dictInputs['mech_terms_select']:
    mechall, Qabc_contrib_dict = spectrumObj.intensity_mechanical()
    sec_hypol_data2 += mechall
    # print(f'Mechanical {dictInputs["mech_terms_select"]}')
    # print(repr(sec_hypol_data2))

sec_hypol_dataALL+=sec_hypol_data2

intensity = abs(sec_hypol_dataALL) ** 2

# print('\nSum both')
# print(repr(sec_hypol_dataALL))

# print('\nIntensity abs(sec_hypol_dataALL)**2')
# print(repr(intensity))

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
from wilson import rendering
import os

# this part is awkward right now; it can be updated according to changes in Spectrum2D class
settings_here = {'electrical': [0,1], 'mechanical': [0,],
                 'Gamma_rc': 10., 'region': 1,
                 'font_dict': {'size': 18}, 'figsize': (12, 15), 'norm_max': None, 'norm_min': None,
                 'dynamic_range_n': 3000}
other = {'regions': {1: ((1280., 3150., 235.41), (1589., 6050., 465.41))}, 'terms_selection': ([0,1], [0,]), 'w1mw2': False, 'log10': True}

name = rendering.make_name(datadict, vib_levels_harmonic=False, settings=settings_here, other=other, directory='.')
artist = rendering.SpectrumFigure(sec_hypol_dataALL, spectrumObj.w1_mesh, spectrumObj.w2_mesh, settings_here)
title_on_top, text_under_the_figure = rendering.make_texts4fig(datadict, spectrumObj, artist, settings_here, other, '.')
fig = artist.plot2Dmatplotlib(nametuple=(name, os.path.join(os.path.dirname('__file__')), title_on_top),
                                text_under_the_figure=text_under_the_figure, diagonal=False, to_save=False)
plt.show()